In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler,normalize
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

In [3]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    return rmse, mae, r2, nse

In [5]:
def process_dataframe(df, target_column, test_size=0.1, random_state=42):
    # List of all possible CH4 columns
    carbon_columns = ['CO2_diffusive__mgCO2_m_2_h_1_',
                   'CH4_diffusive__mgCH4_m_2_h_1_', 
                   'CH4_Bubbling__mgCH4_m_2_h_1_']

    # Drop the columns that are not the target column
    filtered_df = df.drop(columns=[col for col in carbon_columns if col != target_column])
    filtered_df = filtered_df.drop(columns=['GSD_ID','Climate_zone','Reservoir_name','Impoundment_year', 'GRanD_ID','Period_of_sampling',
       'Longitude__DD_', 'Latitude__DD_','Dam_Height__m_','Catchment_area__km2_','Chl a','GHI_kWh_m2_month','PET_kg_m_2_month_1','CH4_Total_Flux_mg_m_2_h_1_',
                      'Total_Phosphorus__mg_L_1_','Sampled_Year','Source', 'Sampled_month',
                      'References', 'Ref__no', 'Final_Ref_no', 'SOC', 'Region'])
    filtered_df = filtered_df.dropna()
    # Columns to log transform
    log_transform_columns = ['Volume_million_m3', 'Reservoir_area__km2_','CH4_Bubbling__mgCH4_m_2_h_1_']

    # Apply log transformation to specified columns
    for column in log_transform_columns:
        if column in filtered_df.columns:
            filtered_df[f'log_{column}'] = np.log10(filtered_df[column]) 
            filtered_df.drop(columns=[column], inplace=True)

    # Update the target column name if it was log-transformed
    if target_column in log_transform_columns:
        target_column = f'log_{target_column}'


    X = filtered_df.drop(columns=[target_column])
    y = filtered_df[target_column]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test

In [7]:
# 加载数据
df_raw = pd.read_csv('Data\merged_df_v2.csv')

In [9]:
def grid_search_function (param_grid, model, X_train_scaled, X_test_scaled, y_train, y_test):
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=1)
    grid_search.fit(X_train_scaled, y_train)


    print("最佳参数: ", grid_search.best_params_)


    best_model = grid_search.best_estimator_



    y_pred_train = best_model.predict(X_train_scaled)
    y_pred_test = best_model.predict(X_test_scaled)

    train_rmse, train_mae, train_r2, train_nse = evaluate(y_train, y_pred_train)
    print("Training Set")
    print(f"RMSE: {train_rmse}")
    print(f"MAE: {train_mae}")
    print(f"R^2: {train_r2}")
    print(f"NSE: {train_nse}")
    

    rmse, mae, r2, nse = evaluate(y_test, y_pred_test)

    print("Testing set")
    print(f"RMSE: {rmse}")
    print(f"MAE: {mae}")
    print(f"R^2: {r2}")
    print(f"NSE: {nse}")

In [11]:
CO2_X_train_scaled, CO2_X_test_scaled, CO2_y_train, CO2_y_test = process_dataframe(df_raw, 'CO2_diffusive__mgCO2_m_2_h_1_')
CH4_X_train_scaled, CH4_X_test_scaled, CH4_y_train, CH4_y_test = process_dataframe(df_raw, 'CH4_diffusive__mgCH4_m_2_h_1_')
ebu_CH4_X_train_scaled, ebu_CH4_X_test_scaled, ebu_CH4_y_train, ebu_CH4_y_test = process_dataframe(df_raw, 'CH4_Bubbling__mgCH4_m_2_h_1_')

## CO2_diffusive

### RF

In [13]:
param_grid_rf_CO2_diff = {
    'n_estimators': [180,190,200,210,220],
    'max_depth': [None, 9, 10, 11, 12],
    'min_samples_split': [15, 20, 25, 30],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

In [ ]:

rf = RandomForestRegressor(random_state=42)


grid_search_rf = GridSearchCV(rf, param_grid_rf_CO2_diff, cv=5, scoring='r2')

grid_search_rf.fit(CO2_X_train_scaled, CO2_y_train)

best_rf = grid_search_rf.best_estimator_



y_pred_train = best_rf.predict(CO2_X_train_scaled)
y_pred_rf = best_rf.predict(CO2_X_test_scaled)

#results = grid_search_rf.cv_results_
#for mean_score, params in zip(results['mean_test_score'], results['params']):
    #print(f"Mean R^2: {mean_score:.4f} with params: {params}")

rmse_rf, mae_rf, r2_rf, nse_rf = evaluate(CO2_y_test, y_pred_rf)

print(grid_search_rf.best_params_)
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R^2: {r2_rf:.4f}")
print(f"NSE: {nse_rf:.4f}")


### XGB

In [ ]:
xgb = XGBRegressor(random_state=42)


param_grid_xgb_CO2_diff = {
    'n_estimators': [180, 190, 200, 210, 220],
    'max_depth': [7,8,9,10],
    'learning_rate': [0.001,0.01,0.1],
    'subsample': [0.3,0.4,0.5,0.6,0.7]
}


grid_search_xgb = GridSearchCV(xgb, param_grid_xgb_CO2_diff, cv=5, scoring='r2')

grid_search_xgb.fit(CO2_X_train_scaled, CO2_y_train)


best_xgb = grid_search_xgb.best_estimator_


y_pred_train_xgb = best_xgb.predict(CO2_X_train_scaled)
y_pred_xgb = best_xgb.predict(CO2_X_test_scaled)


rmse_xgb, mae_xgb, r2_xgb, nse_xgb = evaluate(CO2_y_test, y_pred_xgb)


print(grid_search_xgb.best_params_)
print(f"RMSE: {rmse_xgb:.4f}")
print(f"MAE: {mae_xgb:.4f}")
print(f"R^2: {r2_xgb:.4f}")
print(f"NSE: {nse_xgb:.4f}")

### KNN

In [ ]:
knn = KNeighborsRegressor()

knn_param_grid = {
    'n_neighbors':[i for i in range(20,40)], 
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
grid_search_function(knn_param_grid,knn,CO2_X_train_scaled, CO2_X_test_scaled, CO2_y_train, CO2_y_test)

### SVM

In [ ]:
from sklearn.svm import SVR

svr = SVR()

svr_param_space = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'C': [0.1, 1, 10, 100, 1000, 10000], 
    'epsilon': (0.01, 0.5)  
}
grid_search_function(svr_param_space,svr,CO2_X_train_scaled, CO2_X_test_scaled, CO2_y_train, CO2_y_test)

## CH4_diffusive

#### RF

In [17]:
param_grid_rf_CH4_diff = {
    'n_estimators': [100,110,120,130],
    'max_depth': [None, 12,13,14,15],
    'min_samples_split': [1, 2, 3, 4],
    'min_samples_leaf': [1, 2, 3],
    'bootstrap': [True, False]
}

In [ ]:
rf = RandomForestRegressor(random_state=42)

grid_search_rf = GridSearchCV(rf, param_grid_rf_CH4_diff, cv=5, n_jobs=1, scoring='r2')

grid_search_rf.fit(CH4_X_train_scaled, CH4_y_train)

best_rf = grid_search_rf.best_estimator_

y_pred_rf = best_rf.predict(CH4_X_test_scaled)

rmse_rf, mae_rf, r2_rf, nse_rf = evaluate(CH4_y_test, y_pred_rf)

print(grid_search_rf.best_params_)
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R^2: {r2_rf:.4f}")
print(f"NSE: {nse_rf:.4f}")

### XGB

In [ ]:
xgb = XGBRegressor(random_state=42)

param_grid_xgb_CH4_diff = {
    'n_estimators': [50,60,70,80],
    'max_depth': [11,12,13,14],
    'learning_rate': [0.001,0.01,0.1],
    'subsample': [0.1,0.2,0.3,0.4]
}

grid_search_CH4_xgb = GridSearchCV(xgb, param_grid_xgb_CH4_diff, cv=5, scoring='r2')

grid_search_CH4_xgb.fit(CH4_X_train_scaled, CH4_y_train)

best_CH4_xgb = grid_search_CH4_xgb.best_estimator_

CH4_y_pred_train_xgb = best_CH4_xgb.predict(CH4_X_train_scaled)
CH4_y_pred_xgb = best_CH4_xgb.predict(CH4_X_test_scaled)

rmse_xgb, mae_xgb, r2_xgb, nse_xgb = evaluate(CH4_y_test, CH4_y_pred_xgb)

print(grid_search_CH4_xgb.best_params_)
print(f"RMSE: {rmse_xgb:.4f}")
print(f"MAE: {mae_xgb:.4f}")
print(f"R^2: {r2_xgb:.4f}")
print(f"NSE: {nse_xgb:.4f}")

### KNN

In [ ]:
knn_param_grid = {
    'n_neighbors': [i for i in range(60,80)],  # 1到30的整数范围
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
grid_search_function(knn_param_grid,knn,CH4_X_train_scaled, CH4_X_test_scaled, CH4_y_train, CH4_y_test)

### SVM

In [ ]:
svr_param_space = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'C': [0.1, 1, 10, 100, 1000, 10000],  # C的范围为0.1到100，使用对数均匀分布
    'epsilon':(0.01, 0.5)  # epsilon的范围为0.01到0.5
}
grid_search_function(svr_param_space,svr,CH4_X_train_scaled, CH4_X_test_scaled, CH4_y_train, CH4_y_test)

## CH4 Ebullitive

### RF

In [59]:
param_grid_rf_CH4_ebu = {
    'n_estimators': [120,130,140,150],
    'max_depth': [None, 10,11,12],
    'min_samples_split': [1, 2, 3],
    'min_samples_leaf': [1, 2, 3],
    'bootstrap': [True, False]
}

In [ ]:
rf = RandomForestRegressor(random_state=42)

grid_search_rf = GridSearchCV(rf, param_grid_rf_CH4_ebu, cv=5, n_jobs=1, scoring='r2')

grid_search_rf.fit(ebu_CH4_X_train_scaled, ebu_CH4_y_train)

best_rf = grid_search_rf.best_estimator_



y_pred_rf = best_rf.predict(ebu_CH4_X_test_scaled)

rmse_rf, mae_rf, r2_rf, nse_rf = evaluate(ebu_CH4_y_test, y_pred_rf)

print(grid_search_rf.best_params_)
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R^2: {r2_rf:.4f}")
print(f"NSE: {nse_rf:.4f}")

### XGB

In [ ]:
xgb = XGBRegressor(random_state=42)

param_grid_xgb_ebu = {
    'n_estimators': [90,100,110,120],
    'max_depth': [5,6,7,8,9],
    'learning_rate': [0.01,0.1,0.2,0.3],
    'subsample': [0.1,0.2,0.3,0.4]
}

grid_search_ebu_CH4_xgb = GridSearchCV(xgb, param_grid_xgb_ebu, cv=5, scoring='r2')

grid_search_ebu_CH4_xgb.fit(ebu_CH4_X_train_scaled, ebu_CH4_y_train)

best_ebu_CH4_xgb = grid_search_ebu_CH4_xgb.best_estimator_

ebu_CH4_y_pred_train_xgb = best_ebu_CH4_xgb.predict(ebu_CH4_X_train_scaled)
ebu_CH4_y_pred_xgb = best_ebu_CH4_xgb.predict(ebu_CH4_X_test_scaled)

rmse_xgb, mae_xgb, r2_xgb, nse_xgb = evaluate(ebu_CH4_y_test, ebu_CH4_y_pred_xgb)

print(grid_search_ebu_CH4_xgb.best_params_)
print(f"RMSE: {rmse_xgb:.4f}")
print(f"MAE: {mae_xgb:.4f}")
print(f"R^2: {r2_xgb:.4f}")
print(f"NSE: {nse_xgb:.4f}")

### KNN

In [ ]:
knn_param_grid = {
    'n_neighbors':[i for i in range(5,20)],  # 1到30的整数范围
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
grid_search_function(knn_param_grid,knn,ebu_CH4_X_train_scaled, ebu_CH4_X_test_scaled, ebu_CH4_y_train, ebu_CH4_y_test)

### SVM

In [ ]:
# 定义贝叶斯优化的参数空间
svr_param_space = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'C': [0.1, 1, 10, 100, 1000, 10000],  # C的范围为0.1到100，使用对数均匀分布
    'epsilon': (0.01, 0.5) # epsilon的范围为0.01到0.5
}
grid_search_function(svr_param_space,svr,ebu_CH4_X_train_scaled, ebu_CH4_X_test_scaled, ebu_CH4_y_train, ebu_CH4_y_test)

## CH4 Degassing

In [ ]:
df_gres = pd.read_csv("Data/G-res Database.csv")

In [ ]:
def elastic_grid_search(X_train_scaled, X_test_scaled,y_train, y_test, carbon_columns,params_number):
    param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0.1,0.2, 0.3,0.4, 0.5,0.6, 0.7,0.8, 0.9, 1.0]}
    elastic_net = ElasticNet()
    grid_search = GridSearchCV(estimator=elastic_net, param_grid=param_grid, cv=5, scoring='r2')
    grid_search.fit(X_train_scaled, y_train)

    # Best parameters
    print("Best alpha:", grid_search.best_params_['alpha'])
    print("Best l1_ratio:", grid_search.best_params_['l1_ratio'])

    # Fit the model with the best parameters
    best_elastic_net= grid_search.best_estimator_
    best_elastic_net.fit(X_train_scaled, y_train)

    # Predict on the test set
    y_pred = best_elastic_net.predict(X_test_scaled)
    
    y_train_pred = best_elastic_net.predict(X_train_scaled)
    
    
    print("Training set")
    #print(f"RMSE: {train_rmse}")
    #print(f"R^2: {r2_score(10**y_train, 10**y_train_pred)}")
    print(f"R^2: {r2_score(y_train, y_train_pred)}")
    print("Testing set")
    #print(f"RMSE: {rmse}")
    #print(f"R^2: {r2_score(10**y_test, 10**y_pred)}")
    print(f"R^2: {r2_score(y_test, y_pred)}")
    print(carbon_columns)
    # Get the coefficients of the trained ElasticNet model
    coefficients= best_elastic_net.coef_
    # Display the coefficients
    print("Coefficients of the ElasticNet model:", coefficients)
    print("Intercept of the ElasticNet model:",best_elastic_net.intercept_)
    #print('Stand error of residuals:', s)
    return best_elastic_net

df_gres['annual_runoff_mm_yr'] = df_gres['discharge__m3_s_1_']*60*60*24*365*df_gres['Catchment area (km2)']
df_gres['Water Residence Time (WRT, yrs)']=((df_gres['Mean_depth__m_']*df_gres['Reservoir_area__km2_'])/(df_gres['Catchment area (km2)']*df_gres['annual_runoff_mm_yr']))*1000
df_gres['Age'] = df_gres['Year of samping']-df_gres['Impoundment year']
df_gres.loc[df_gres['Age']<=0,'Age']=0.5
deg_target_column = 'Difference CH4 concentration reservoir intake-outlet (mgC L-1)'
deg_filtered_df = df_gres.drop(columns=['Reservoir name', 'GRanD ID', 'Impoundment year', 'Longitude (DD)',
       'Latitude (DD)', 'Climate zone ','Littoral area (%)', 'River area before impoundment (%)',
       'Catchment area (km2)', 'Maximum depth  (m)', 'Thermocline depth (m)', 'Discharge (m3 s-1)', 'WRT (yrs)',
       'Total Phosphorus (µg L-1)', 'Trophic Level','Soil Carbon Content (kgC m-2)', 'Cumulative radiance (kWh m-2 period)',
       'Effective temperature CH4 (°C)', 'Effective temperature CO2 (°C)', 'annual_runoff_mm_yr', 'Water Residence Time (WRT, yrs)',
       'References', 'Year of samping', 'Period of sampling','Annual Corrected CH4 diffusive (mgC m-2 d-1)',
        'Annual Corrected CH4 Bubbling (mgC m-2 d-1)','Annual Corrected CO2 diffusive (mgC m-2 d-1)', 'Outliers', 'Degassing',
       'Degassing unit','PET_kg_m_2_month_1','GHI_kWh_m2_month','SOC']).dropna()
deg_filtered_df  = deg_filtered_df[['Mean_depth__m_','Temperature____', 'Wind_speed__m_s_1_','Age','Precipitation_kg_m_2_month',
    'discharge__m3_s_1_', 'VPD_Pa','HURS_%','NPP_g_C_m_2_yr','rsds_MJ_m2_d','Volume_million_m3','Reservoir_area__km2_','Difference CH4 concentration reservoir intake-outlet (mgC L-1)']]
# Columns to log transform
deg_log_transform_columns = ['Volume_million_m3','Reservoir_area__km2_','Difference CH4 concentration reservoir intake-outlet (mgC L-1)']

# Apply log transformation to specified columns
for column in deg_log_transform_columns:
    if column in deg_filtered_df.columns:
        deg_filtered_df[f'log_{column}'] = np.log10(deg_filtered_df[column]) 
        deg_filtered_df.drop(columns=[column], inplace=True)

# Update the target column name if it was log-transformed
if deg_target_column in deg_log_transform_columns:
    deg_target_column = f'log_{deg_target_column}'


deg_X = deg_filtered_df.drop(columns=[deg_target_column])
print(deg_X.columns)
deg_y = deg_filtered_df[deg_target_column]
# 划分数据集
deg_X_train, deg_X_test, deg_y_train, deg_y_test = train_test_split(deg_X, deg_y, test_size=0.1, random_state=42)
deg_label_df = df_gres.copy()
deg_train_indices = deg_X_train.index
deg_test_indices = deg_X_test.index
deg_label_df['split'] = 'train'  # Default to 'train'
deg_label_df.loc[deg_test_indices, 'split'] = 'test'

# 数据归一化
deg_scaler = MinMaxScaler(feature_range=(0, 1))
deg_X_train_scaled = deg_scaler.fit_transform(deg_X_train)
deg_X_test_scaled = deg_scaler.transform(deg_X_test)
deg_elastic_net = elastic_grid_search(deg_X_train_scaled, deg_X_test_scaled, deg_y_train, deg_y_test, 'CH4_degassing__mgC_L_',5)